# REPRODUCE BASELINE EXPERIMENT: GRAZPEDWRI-DX DATASET (YOLOv5m)
**Bài báo gốc:** *A pediatric wrist trauma X-ray dataset (GRAZPEDWRI-DX) for machine learning* (Nature Scientific Data, 2022)

### Mục tiêu thực nghiệm:
- Tái lập thực nghiệm Object Detection với mô hình **YOLOv5m** (COCO pre-trained).
- Huấn luyện phân tán trên **2 GPU Tesla T4 (DDP - Distributed Data Parallel)** trên Kaggle.
- Chia tập dữ liệu: **15.327 Train / 4.000 Validation / 1.000 Test**.
- Đánh giá trên tập Test để đối chiếu trực tiếp với **Bảng 3** của bài báo (mục tiêu nhãn `fracture` mAP@0.5 ~ 0.933).

## 1. Kiểm tra môi trường phần cứng (2 x GPU T4)

In [ ]:
import torch
import os

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Number of GPUs: {gpu_count}")
for i in range(gpu_count):
    print(f"  - GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

assert gpu_count >= 1, "Cần bật Accelerator GPU trong Settings của Kaggle Notebook!"

## 2. Cài đặt YOLOv5 từ mã nguồn chính thức (Ultralytics)

In [ ]:
%cd /kaggle/working
!rm -rf yolov5
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -qr requirements.txt
print("YOLOv5 đã sẵn sàng!")

## 3. Định vị và chuẩn bị dữ liệu GRAZPEDWRI-DX
Đoạn mã dưới đây tự động nhận diện dữ liệu trong `/kaggle/input`:
- **Trường hợp A:** Bạn đã bấm **Add Input** dùng Dataset có sẵn trên Kaggle (`grazpedwri-dx`).
- **Trường hợp B:** Bạn upload file ZIP tải từ Figshare (chứa `images_part1..4.zip` và `folder_structure.zip`).

In [ ]:
import os
import glob
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path("/kaggle/working/dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra thư mục input
input_files = list(Path("/kaggle/input").rglob("*"))
print(f"Tìm thấy {len(input_files)} tệp tin trong /kaggle/input")

# Kiểm tra xem có file zip dạng parts (images_part1.zip...) không
zip_parts = list(Path("/kaggle/input").rglob("images_part*.zip"))
folder_struct_zips = list(Path("/kaggle/input").rglob("folder_structure.zip"))

if zip_parts:
    print("Phát hiện dữ liệu dạng file nén chia phần từ Figshare. Đang giải nén...")
    img_out = DATA_DIR / "images"
    img_out.mkdir(exist_ok=True)
    for zp in sorted(zip_parts):
        print(f"Giải nén {zp.name}...")
        with zipfile.ZipFile(zp, 'r') as z:
            z.extractall(img_out)
            
    if folder_struct_zips:
        print(f"Giải nén {folder_struct_zips[0].name}...")
        with zipfile.ZipFile(folder_struct_zips[0], 'r') as z:
            z.extractall(DATA_DIR)
else:
    # Dữ liệu đã được giải nén sẵn trong Kaggle Dataset
    print("Dữ liệu đã có sẵn trong Kaggle input. Đang tìm kiếm thư mục ảnh và nhãn...")
    png_files = list(Path("/kaggle/input").rglob("*.png"))
    print(f"Tìm thấy {len(png_files)} ảnh PNG trong /kaggle/input")

## 4. Phân chia tập dữ liệu (15.327 Train / 4.000 Val / 1.000 Test)
Theo đúng phân bổ tác giả công bố trong bài báo.

In [ ]:
import random
import yaml

# Tìm đường dẫn chứa toàn bộ ảnh PNG và nhãn TXT (định dạng YOLO)
all_pngs = sorted(list(DATA_DIR.rglob("*.png")) or list(Path("/kaggle/input").rglob("*.png")))
print(f"Tổng số ảnh tìm thấy: {len(all_pngs)}")

# Đọc danh sách file và tạo danh sách đường dẫn
random.seed(42)
shuffled_images = all_pngs.copy()
random.shuffle(shuffled_images)

# Phân chia đúng theo bài báo
train_imgs = shuffled_images[:15327]
val_imgs = shuffled_images[15327:15327 + 4000]
test_imgs = shuffled_images[15327 + 4000:15327 + 4000 + 1000]

print(f"Train set: {len(train_imgs)} ảnh")
print(f"Validation set: {len(val_imgs)} ảnh")
print(f"Test set: {len(test_imgs)} ảnh")

# Lưu danh sách file txt cho YOLO
with open('/kaggle/working/train.txt', 'w') as f:
    for p in train_imgs:
        f.write(f"{str(p.resolve())}\n")

with open('/kaggle/working/val.txt', 'w') as f:
    for p in val_imgs:
        f.write(f"{str(p.resolve())}\n")

with open('/kaggle/working/test.txt', 'w') as f:
    for p in test_imgs:
        f.write(f"{str(p.resolve())}\n")

# Danh sách 9 lớp đối tượng theo định dạng GRAZPEDWRI-DX
classes = [
    "boneanomaly",
    "bonelesion",
    "foreignbody",
    "fracture",
    "metal",
    "periostealreaction",
    "pronatorsign",
    "softtissue",
    "text"
]

data_yaml_content = {
    'path': '/kaggle/working',
    'train': 'train.txt',
    'val': 'val.txt',
    'test': 'test.txt',
    'nc': len(classes),
    'names': classes
}

with open('/kaggle/working/yolov5/data/grazpedwri.yaml', 'w') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

print("File cấu hình data/grazpedwri.yaml đã được tạo thành công!")

## 5. Huấn luyện Baseline YOLOv5m trên 2 GPU (DDP)
Chạy lệnh huấn luyện với:
- Model: `yolov5m.pt` (COCO pre-trained)
- Batch size: `16` (chia đều 8 ảnh/GPU)
- Image size: `640`
- Epochs: `50`
- Multi-GPU: `device 0,1` qua `torch.distributed.run`

In [ ]:
%cd /kaggle/working/yolov5

!python -m torch.distributed.run --nproc_per_node 2 train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data data/grazpedwri.yaml \
    --weights yolov5m.pt \
    --device 0,1 \
    --project /kaggle/working/runs \
    --name yolov5m_baseline

## 6. Đánh giá trên tập TEST (1.000 ảnh) và Đối chiếu Bảng 3 bài báo

In [ ]:
%cd /kaggle/working/yolov5

# Chạy đánh giá trên tập test bằng trọng số tốt nhất best.pt
!python val.py \
    --weights /kaggle/working/runs/yolov5m_baseline/weights/best.pt \
    --data data/grazpedwri.yaml \
    --img 640 \
    --task test \
    --device 0

### Bảng tham chiếu kết quả bài báo gốc (Table 3, Scientific Data 2022):

| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
| :--- | :--- | :--- | :--- | :--- |
| **fracture (gãy xương)** | **0.917** | **0.887** | **0.933** | **0.544** |
| metal | 0.918 | 1.000 | 0.986 | 0.777 |
| text | 0.975 | 0.987 | 0.992 | 0.731 |
| pronatorsign | 0.835 | 0.656 | 0.820 | 0.378 |
| periostealreaction | 0.713 | 0.640 | 0.675 | 0.317 |
| boneanomaly | 0.633 | 0.267 | 0.386 | 0.158 |
| softtissue | 0.397 | 0.220 | 0.302 | 0.139 |
| foreignbody | 1.000 | 0.000 | 0.496 | 0.446 |
| bonelesion | 0.000 | 0.000 | 0.012 | 0.009 |
| **Total (Trung bình)** | **0.710** | **0.517** | **0.622** | **0.389** |